# Рекуррентные нейронные сети

**AI3 · NLP Deep Dive** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

В предыдущих разделах мы использовали богатые семантические представления текста и простой линейный классификатор поверх эмбеддингов. Такая архитектура позволяет захватывать агрегированное значение слов в предложении, но не учитывает **порядок** слов, так как операция агрегации поверх эмбеддингов удаляет эту информацию из исходного текста. Поскольку такие модели не могут моделировать порядок слов, они не способны решать более сложные или неоднозначные задачи, такие как генерация текста или ответы на вопросы.

Чтобы захватить смысл последовательности текста, необходимо использовать другую архитектуру нейронной сети, которая называется **рекуррентной нейронной сетью** или RNN. В RNN мы пропускаем предложение через сеть по одному символу за раз, и сеть производит некоторое **состояние**, которое затем передается обратно в сеть вместе со следующим символом.

![RNN](ms-assets/ai/rnn.27f5c29c53d727b5.webp)

> Изображение автора

Для входной последовательности токенов X<sub>0</sub>,...,X<sub>n</sub> RNN создает последовательность блоков нейронной сети и обучает эту последовательность от начала до конца с использованием обратного распространения ошибки. Каждый блок сети принимает пару (X<sub>i</sub>,S<sub>i</sub>) в качестве входных данных и производит S<sub>i+1</sub> в качестве результата. Финальное состояние S<sub>n</sub> или (выход Y<sub>n</sub>) передается в линейный классификатор для получения результата. Все блоки сети имеют одинаковые веса и обучаются от начала до конца с использованием одного прохода обратного распространения.

Благодаря тому, что векторы состояния S<sub>0</sub>,...,S<sub>n</sub> передаются через сеть, она способна изучать последовательные зависимости между словами. Например, если слово *not* появляется где-то в последовательности, сеть может научиться отрицать определенные элементы внутри вектора состояния, что приводит к отрицанию.

> ✅ Поскольку веса всех блоков RNN на изображении выше являются общими, то ту же схему можно представить как один блок (справа) с рекуррентной обратной связью, которая передает выходное состояние сети обратно на вход.



## Анатомия ячейки RNN

Давайте рассмотрим, как устроена простая ячейка RNN. Она принимает предыдущее состояние S<sub>i-1</sub> и текущий символ X<sub>i</sub> в качестве входных данных и должна произвести выходное состояние S<sub>i</sub> (а иногда нас также интересует другой выход Y<sub>i</sub>, как в случае с генеративными сетями).

Простая ячейка RNN имеет две матрицы весов внутри: одна преобразует входной символ (назовем ее W), а другая преобразует входное состояние (H). В этом случае выход сети вычисляется как σ(W×X<sub>i</sub>+H×S<sub>i-1</sub>+b), где σ — это функция активации, а b — дополнительное смещение.

![Анатомия ячейки RNN](ms-assets/ai/rnn-anatomy.79ee3f3920b3294b.webp)

> Изображение автора

Во многих случаях входные токены проходят через слой эмбеддинга перед тем, как попасть в RNN, чтобы уменьшить размерность. В этом случае, если размерность входных векторов равна *emb_size*, а вектора состояния — *hid_size*, то размер W будет *emb_size*×*hid_size*, а размер H — *hid_size*×*hid_size*.



## Долгая краткосрочная память (LSTM)

Одна из основных проблем классических RNN — это проблема **затухающих градиентов**. Поскольку RNN обучаются от начала до конца в одном проходе обратного распространения, им сложно передавать ошибку к первым слоям сети, и, следовательно, сеть не может изучать отношения между удаленными токенами. Один из способов избежать этой проблемы — ввести **явное управление состоянием** с помощью так называемых **врат**. Существуют две хорошо известные архитектуры такого типа: **долгая краткосрочная память** (LSTM) и **управляемый рекуррентный блок** (GRU).

![Пример ячейки долгой краткосрочной памяти](ms-assets/ai/lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

> Источник изображения TBD

Сеть LSTM организована аналогично RNN, но через слои передаются два состояния: фактическое состояние C и скрытый вектор H. В каждой единице скрытый вектор H<sub>i</sub> объединяется с входом X<sub>i</sub>, и они управляют тем, что происходит с состоянием C через **врата**. Каждые врата представляют собой нейронную сеть с сигмоидной активацией (выход в диапазоне [0,1]), которую можно рассматривать как побитовую маску при умножении на вектор состояния. Существуют следующие врата (слева направо на изображении выше):

* **Врата забывания** принимают скрытый вектор и определяют, какие компоненты вектора C нужно забыть, а какие пропустить.
* **Входные врата** берут некоторую информацию из входного и скрытого векторов и вставляют ее в состояние.
* **Выходные врата** преобразуют состояние через линейный слой с активацией *tanh*, затем выбирают некоторые из его компонентов с помощью скрытого вектора H<sub>i</sub>, чтобы произвести новое состояние C<sub>i+1</sub>.

Компоненты состояния C можно рассматривать как флаги, которые можно включать и выключать. Например, если мы встречаем имя *Alice* в последовательности, мы можем предположить, что оно относится к женскому персонажу, и установить флаг в состоянии, что в предложении есть существительное женского рода. Если далее мы встречаем фразу *and Tom*, мы установим флаг, что у нас есть множественное число существительных. Таким образом, манипулируя состоянием, мы можем предположительно отслеживать грамматические свойства частей предложения.

> ✅ Отличный ресурс для понимания внутреннего устройства LSTM — это замечательная статья [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) Кристофера Олаха.



## Двунаправленные и многослойные RNN

Мы обсудили рекуррентные сети, которые работают в одном направлении — от начала последовательности к концу. Это выглядит естественно, так как напоминает способ, которым мы читаем и слушаем речь. Однако, поскольку во многих практических случаях у нас есть случайный доступ к входной последовательности, имеет смысл выполнять рекуррентные вычисления в обоих направлениях. Такие сети называются **двунаправленными** RNN. При работе с двунаправленной сетью нам понадобятся два скрытых вектора состояния — по одному для каждого направления.

Рекуррентная сеть, будь то однонаправленная или двунаправленная, захватывает определенные шаблоны внутри последовательности и может сохранять их в векторе состояния или передавать в выход. Как и в случае с сверточными сетями, мы можем построить еще один рекуррентный слой поверх первого, чтобы захватывать шаблоны более высокого уровня и строить их из шаблонов низкого уровня, извлеченных первым слоем. Это приводит нас к понятию **многослойной RNN**, которая состоит из двух или более рекуррентных сетей, где выход предыдущего слоя передается следующему слою в качестве входа.

![Изображение многослойной LSTM RNN](ms-assets/ai/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Картинка из [этого замечательного поста](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) Фернандо Лопеса*



## Упражнения: Эмбеддинги

Продолжите обучение в следующих ноутбуках:

* [RNNs с PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/16-RNN/RNNPyTorch.ipynb)
* [RNNs с TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/16-RNN/RNNTF.ipynb)



## Заключение

В этом разделе мы увидели, что RNN могут использоваться для классификации последовательностей, но на самом деле они могут решать гораздо больше задач, таких как генерация текста, машинный перевод и многое другое. Мы рассмотрим эти задачи в следующем разделе.



## Задание

Прочитайте литературу о LSTM и подумайте об их применении:

- [Grid Long Short-Term Memory](https://arxiv.org/pdf/1507.01526v1.pdf)
- [Show, Attend and Tell: Neural Image Caption
Generation with Visual Attention](https://arxiv.org/pdf/1502.03044v2.pdf)



## Обзор и самостоятельное изучение

- [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) Кристофера Олаха.



---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/16-RNN)
- [Учебник ML (ШАД): нейросети для последовательностей](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/neural-networks-for-sequences)